<b style="color: cyan; font-size: 20px;">Forward Pass</b>


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()

        #Defining Layers
        self.fc1 = nn.Linear(10,5)
        self.fc2 = nn.Linear(5, 2)


    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x
#Creating a Module
model = SimpleNet()

dummy_input = torch.randn(1,10)

prediction = model(dummy_input)
print(f"Input: {dummy_input}")
print(f"Prediction: {prediction}")
        


Input: tensor([[-1.0479, -0.0359, -1.3392, -0.2131,  0.4716,  1.0140,  0.2105,  0.2760,
         -0.2319, -0.8507]])
Prediction: tensor([[-0.5466, -0.1225]], grad_fn=<AddmmBackward0>)


<b style="color: cyan; font-size: 20px;">
Loss computation,
Backward pass,
Optimizer step
Zeroing gradients
 </b>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

x_train = torch.randn(100, 1)
y_actual = 3*x_train + 2+ torch.randn(100, 1)*0.1


model = nn.Linear(1,1)

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

#Training loop
for epoch in range(10):
    y_pred = model(x_train)

    #Computer Loss
    loss = criterion(y_pred, y_actual)

    #Backward passs
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f'Epoch [{epoch+1}/10], Loss: {loss.item():.4f}')





Epoch [1/10], Loss: 12.0937
Epoch [2/10], Loss: 11.6699
Epoch [3/10], Loss: 11.2609
Epoch [4/10], Loss: 10.8664
Epoch [5/10], Loss: 10.4857
Epoch [6/10], Loss: 10.1183
Epoch [7/10], Loss: 9.7639
Epoch [8/10], Loss: 9.4219
Epoch [9/10], Loss: 9.0919
Epoch [10/10], Loss: 8.7735


<b style="color: cyan; font-size: 20px;">
Validation loop
 </b>

A validation loop in PyTorch is a separate pass through your "unseen" validation data to check how well your model generalizes. Unlike the training loop, it does not update the model's weights. 
The Essential Steps
A proper validation loop follows these three critical rules:

***model.eval():*** Switches layers like Dropout and Batch Normalization into evaluation mode so they behave consistently.

***torch.no_grad():*** A context manager that disables gradient calculation, which saves memory and speeds up computation since we aren't doing backpropagation.

***No Optimizer:*** You only perform the forward pass and loss calculation; never call optimizer.step(). 
Standard Validation Loop Code



In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import LabelEncoder


# ── Dataset ──────────────────────────────────────────────────────────────────
class IrisDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.X = torch.tensor(df.iloc[:, :-1].values, dtype=torch.float32)
        le = LabelEncoder()
        self.y = torch.tensor(le.fit_transform(df.iloc[:, -1]), dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# ── Model ─────────────────────────────────────────────────────────────────────
class IrisClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 3),
        )

    def forward(self, x):
        return self.net(x)


# ── Helpers ───────────────────────────────────────────────────────────────────
def run_epoch(model, loader, criterion, optimizer, device, training: bool):
    """Single pass over a DataLoader. Returns (avg_loss, accuracy %)."""
    model.train() if training else model.eval()

    total_loss = 0.0          # ← reset every call, not shared across epochs
    correct    = 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for data, target in loader:
            data, target = data.to(device), target.to(device)

            if training:
                optimizer.zero_grad()

            output = model(data)
            loss   = criterion(output, target)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            correct    += (output.argmax(dim=1) == target).sum().item()

    n        = len(loader.dataset)
    avg_loss = total_loss / len(loader)
    accuracy = 100.0 * correct / n
    return avg_loss, accuracy


# ── Setup ─────────────────────────────────────────────────────────────────────
file_path = r"C:\Users\user\Desktop\Torch_Practices\Py_Torch_Practices\IRIS.csv"

full_dataset = IrisDataset(file_path)

# Proper train / val split (80 / 20)
val_size   = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_set, val_set = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=16, shuffle=False)

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = IrisClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


# ── Training loop ─────────────────────────────────────────────────────────────
epochs = 100
for epoch in range(1, epochs + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device, training=True)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion, optimizer, device, training=False)

    print(
        f"Epoch {epoch:3d}/{epochs} │ "
        f"Train  loss={train_loss:.4f}  acc={train_acc:.1f}%  │ "
        f"Val  loss={val_loss:.4f}  acc={val_acc:.1f}%"
    )

Epoch   1/100 │ Train  loss=1.2299  acc=33.3%  │ Val  loss=1.0306  acc=53.3%
Epoch   2/100 │ Train  loss=0.9629  acc=40.8%  │ Val  loss=0.8858  acc=70.0%
Epoch   3/100 │ Train  loss=0.8506  acc=65.8%  │ Val  loss=0.7180  acc=70.0%
Epoch   4/100 │ Train  loss=0.6551  acc=92.5%  │ Val  loss=0.5944  acc=93.3%
Epoch   5/100 │ Train  loss=0.5429  acc=88.3%  │ Val  loss=0.4864  acc=76.7%
Epoch   6/100 │ Train  loss=0.4612  acc=91.7%  │ Val  loss=0.4313  acc=93.3%
Epoch   7/100 │ Train  loss=0.3985  acc=96.7%  │ Val  loss=0.3917  acc=83.3%
Epoch   8/100 │ Train  loss=0.3650  acc=95.8%  │ Val  loss=0.3579  acc=93.3%
Epoch   9/100 │ Train  loss=0.3176  acc=93.3%  │ Val  loss=0.3314  acc=93.3%
Epoch  10/100 │ Train  loss=0.2835  acc=97.5%  │ Val  loss=0.3111  acc=96.7%
Epoch  11/100 │ Train  loss=0.2668  acc=97.5%  │ Val  loss=0.2935  acc=93.3%
Epoch  12/100 │ Train  loss=0.2274  acc=99.2%  │ Val  loss=0.2783  acc=90.0%
Epoch  13/100 │ Train  loss=0.2016  acc=99.2%  │ Val  loss=0.2610  acc=93.3%